# 8.0 Data Validation And Testing
---
---
This notebook includes additional testing and validation of all notebooks 0 - 8 to keep those clean from clutter. Any changes made relating to the data itself or with regards to analysis techniques and decisions are included here.

#### Test scripts, methods and numerical checks are listed below by section:


---
# 1.0 Data Cleaning and Staging
---
---

---
# 2.0 Data Exploration and Additional Data Rules
---
---

---
# 3.0 Data Segmentation and Customer Table
---
---

### Load Data for testing:

In [1]:
import pandas as pd
import os

# Define the input path for the raw data
interim_parquet_path = "../data/interim/cleansed_online_retail_orders.parquet"
interim_dir = os.path.dirname(interim_parquet_path)

# 1. Check if the required directory structure exists
if not os.path.exists(interim_dir):
    print(f"❌ ERROR: The directory '{interim_dir}' does not exist.")
    print("Action Required - Please set up your local environment:")
    print("  1. Create the standard data folders: 'data/raw/', 'data/interim/', and 'data/processed/'.")
    print("  2. Ensure 'data/' is added to your .gitignore file.")
    print("Once complete, run the entire ETL notebook 1_data_clean_and_stage.")
    print("Then rerun this cell.")

# 2. If directory exists, check if the file already exists
elif not os.path.exists(interim_parquet_path):
    print(f"❌ ERROR: The file '{interim_parquet_path}' does not exist.")
    print("Action Required - Please run the entire ETL notebook 1_data_clean_and_stage.")
    print("Then rerun this cell.")

# 3. If file exists, load to dataframe
else:
    print("⏳ Importing df_cleansed_online_retail_orders ..")
    df_cleansed_online_retail_orders = pd.read_parquet(interim_parquet_path)
    print("✅ Data loaded to dataframe df_cleansed_online_retail_orders complete.")

⏳ Importing df_cleansed_online_retail_orders ..
✅ Data loaded to dataframe df_cleansed_online_retail_orders complete.


---
##### Create dataframe df1 to check total revenue by checkout type untouched and summary df1 to look at it

In [2]:
import pandas as pd

df1 = df_cleansed_online_retail_orders.copy()

# 1. Create Year-Month column 
df1['YearMonth'] = df1['InvoiceDateMin'].dt.to_period('M')

# 2. Group by YearMonth and OrderCheckout
# Aggregate CustomerHashID using 'nunique' and TotalPrice using 'sum'
summary_df1 = df1.groupby(['YearMonth', 'OrderCheckout']).agg({
    'CustomerHashID': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

# 3. Rename the columns for better clarity
summary_df1.columns = ['YearMonth', 'OrderCheckout', 'Distinct_Customers', 'Sum_TotalPrice']

# 4. Convert YearMonth to string
# summary_df1['YearMonth'] = summary_df1['YearMonth'].astype(str)

# 5. Rename the columns
summary_df1.columns = [
    'YearMonth', 
    'OrderCheckout', 
    'Distinct_Customers', 
    'TotalPrice'
]

print(summary_df1.head())

  YearMonth         OrderCheckout  Distinct_Customers  TotalPrice
0   2009-12      CUSTOMER ACCOUNT                1044   663158.55
1   2009-12        GUEST CHECKOUT                   0   136575.06
2   2009-12          TEST ACCOUNT                   0      113.50
3   2009-12  WAREHOUSE ADJUSTMENT                   0        0.00
4   2010-01      CUSTOMER ACCOUNT                 786   531862.90


---
##### Create dataframe summary df2 to check total revenue by checkout type untouched - with other stamps

In [3]:
import pandas as pd

# 1. Group by YearMonth and the three specified categorical columns
# dropna=False is included so orders with 'None' or empty values are still included in sums.
summary_df2 = df1.groupby(
    ['YearMonth', 'OrderCheckout', 'StockCodeType', 'DescriptionType', 'CancellationType'], 
    dropna=False
).agg({
    'CustomerHashID': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

# 2. Rename the columns
summary_df2.columns = [
    'YearMonth', 
    'OrderCheckout', 
    'StockCodeType', 
    'DescriptionType', 
    'CancellationType', 
    'Distinct_Customers', 
    'TotalPrice'
]

# Display the first few rows
print(summary_df2.head())

  YearMonth     OrderCheckout StockCodeType DescriptionType CancellationType  \
0   2009-12  CUSTOMER ACCOUNT  BANK CHARGES    BANK CHARGES     CANCELLATION   
1   2009-12  CUSTOMER ACCOUNT  BANK CHARGES    BANK CHARGES              NaN   
2   2009-12  CUSTOMER ACCOUNT      CARRIAGE        CARRIAGE              NaN   
3   2009-12  CUSTOMER ACCOUNT      DISCOUNT        DISCOUNT     CANCELLATION   
4   2009-12  CUSTOMER ACCOUNT        MANUAL          MANUAL     CANCELLATION   

   Distinct_Customers  TotalPrice  
0                   1      -15.00  
1                   1       15.00  
2                   5      350.00  
3                   5     -108.91  
4                   6    -2908.77  


---
##### Validation 1.0 - check revenue total summary df1 and df2 match
We do a Net Revenue validation on df1 --> summary_df1 --> summary_df2 so we can trust the data prior next steps:

In [4]:
import pandas as pd

# ---------------------------------------------------------
# 1. Calculate total revenue from the original df1
# ---------------------------------------------------------
df1_total_revenue = df1['TotalPrice'].sum()
print(f"Base Revenue (df1): £{df1_total_revenue:,.2f}\n")

# ---------------------------------------------------------
# 2. Create Validation Table for all three DataFrames
# ---------------------------------------------------------
# Calculate the total sums from the summary dataframes
summary1_total_revenue = summary_df1['TotalPrice'].sum()
summary2_total_revenue = summary_df2['TotalPrice'].sum()

# Build validation DataFrame
validation_df = pd.DataFrame({
    'DataSource':['df1', 'summary_df1', 'summary_df2'],
    'TotalNetRevenue':[
        df1_total_revenue, 
        summary1_total_revenue, 
        summary2_total_revenue
    ]
})

# Add formatted column to display the values as £###,###.##
validation_df['TotalNetRevenue_Formatted'] = validation_df['TotalNetRevenue'].apply(lambda x: f"£{x:,.2f}")

# Display final validation table
print("--- Revenue Validation Table ---")
print(validation_df[['DataSource', 'TotalNetRevenue_Formatted']])

# ---------------------------------------------------------
# 3. Do a Check (Return True / False on match)
# ---------------------------------------------------------
# Use round(2) to prevent floating-point errors
revenues =[round(df1_total_revenue, 2), round(summary1_total_revenue, 2), round(summary2_total_revenue, 2)]
all_match = len(set(revenues)) == 1

print("\nTotalNetRevenue values match:", all_match)

Base Revenue (df1): £19,287,250.55

--- Revenue Validation Table ---
    DataSource TotalNetRevenue_Formatted
0          df1            £19,287,250.55
1  summary_df1            £19,287,250.55
2  summary_df2            £19,287,250.55

TotalNetRevenue values match: True


---
##### Export these to data/testing as .csv to facilate further checks in excel

In [ ]:
# output_csv_path1 = '../data/testing/summary_df1.csv'
# output_csv_path2 = '../data/testing/summary_df2.csv'
# print(f"⏳ Exporting test files to testing folder..")
# summary_df1.to_csv(output_csv_path1, index=False)
# summary_df2.to_csv(output_csv_path2, index=False)
# print("✅ Export complete.")

⏳ Exporting test files to testing folder..
✅ Export complete.


---
##### Validation 2.0 - check revenue total summary
Per the excel results check:
- Decision 1 remove Test Account - It looked like a real account with some test transactions. We remove to play safe.
- Decision 2 remove Warehouse Adjustment that has no revenue
- Decision 3 remove Guest Checkout since they are Nan we can't associate their revenue to Customer Accounts

##### We filter on order checkout = Customer Account to produce summary_df3, expected net revenue £16,648,088.87:

In [5]:
import pandas as pd

# 1. Apply customer base filter rules
mask_df1 = (
    # (df_cleansed_online_retail_orders['StockCodeType']    == 'PRODUCT') &
    # (df_cleansed_online_retail_orders['DescriptionType']  == 'PRODUCT') &
    # (df_cleansed_online_retail_orders['CancellationType'].isna()) &
    (df_cleansed_online_retail_orders['OrderCheckout']    == 'CUSTOMER ACCOUNT') # Decision 1, 2, 3 remove test, warehouse and guest checkout
)

df2 = df1.loc[mask_df1].dropna(subset=['CustomerHashID']).copy()

# 3. Group by YearMonth and three categorical columns
# dropna=False is included so orders with 'None' or empty values are still included in sums.
summary_df3 = df2.groupby(
    ['YearMonth', 'OrderCheckout', 'StockCodeType', 'DescriptionType', 'CancellationType'], 
    dropna=False
).agg({
    'CustomerHashID': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

# 4. Rename the columns
summary_df3.columns = [
    'YearMonth', 
    'OrderCheckout', 
    'StockCodeType', 
    'DescriptionType', 
    'CancellationType', 
    'Distinct_Customers', 
    'TotalPrice'
]


# 1. total revenue from df2
df2_total_revenue = df2['TotalPrice'].sum()
print(f"Base Revenue (df2): £{df2_total_revenue:,.2f}\n")

# 2. total revenue from summary_df3
summary_df3_total_revenue = summary_df3['TotalPrice'].sum()
print(f"Base Revenue (summary_df3): £{summary_df3_total_revenue:,.2f}\n")

# output_csv_path3 = '../data/testing/summary_df3.csv'
# print(f"⏳ Exporting test file to testing folder..")
# summary_df3.to_csv(output_csv_path3, index=False)
# print("✅ Export complete.")


# Display the first few rows
print(summary_df3.head())

Base Revenue (df2): £16,648,088.87

Base Revenue (summary_df3): £16,648,088.87

  YearMonth     OrderCheckout StockCodeType DescriptionType CancellationType  \
0   2009-12  CUSTOMER ACCOUNT  BANK CHARGES    BANK CHARGES     CANCELLATION   
1   2009-12  CUSTOMER ACCOUNT  BANK CHARGES    BANK CHARGES              NaN   
2   2009-12  CUSTOMER ACCOUNT      CARRIAGE        CARRIAGE              NaN   
3   2009-12  CUSTOMER ACCOUNT      DISCOUNT        DISCOUNT     CANCELLATION   
4   2009-12  CUSTOMER ACCOUNT        MANUAL          MANUAL     CANCELLATION   

   Distinct_Customers  TotalPrice  
0                   1      -15.00  
1                   1       15.00  
2                   5      350.00  
3                   5     -108.91  
4                   6    -2908.77  


---
##### Validation 3.0 - check revenue  by description type total summary
Per the excel results check:
- Decision 4 summary_df3 is new source of truth aka df2 order table
- Decision 5 remove dec 2011 for 9 days revenue incomplete month, gives 24 months revenue
- Decision 6 remove full invoice re-keys, these are same amount same number same day
- Decision 7 remove line item re-keys, these are same amount same number same day
- Decision 8 - remove bank charges which are financial cost not customer / product value
- Decision 9 - remove non commercial charitable donation 
- Decision 10 - remove carriage logistics costs
- Decision 11 - remove dotcom postage logistics costs
- Decision 12 - remove postage logistics costs

<img src="images/testing1.png" width="1000">

##### We apply decisions 5 - 12 above as mask to make df3 order table and summary_df3, expected net revenue £16,179,795.28:

In [8]:
import pandas as pd

# 1. Apply customer base filter rules
mask_df2 = (
    # Keep these:
    (df1['OrderCheckout'] == 'CUSTOMER ACCOUNT') & # Decision 1, 2, 3 remove test, warehouse and guest checkout
    (df1['InvoiceDateMin'] < '2011-12-01') & # Decisions 4, 5 # remove Dec 2011 
    
    # Exclude these:
    (~df1['CancellationType'].isin(['FULL INVOICE RE-KEY','LINE ITEM RE-KEY'])) & # Decisions 6, 6 # remove Re-Keys 
    (~df1['DescriptionType'].isin(['BANK CHARGES','CANCER RESEARCH COMMISSION'])) & # Decisions 8, 9 # remove Bank Charges and Charitable Donations 
    (~df1['DescriptionType'].isin(['CARRIAGE','DOTCOM POSTAGE','POSTAGE'])) # Decisions 10, 11, 12 # remove Logistics Charges
)

# Apply mask to create new dataframe
df3 = df1.loc[mask_df2].dropna(subset=['CustomerHashID']).copy()

# 3. Group by YearMonth and two categorical columns
# dropna=False is included so orders with 'None' or empty values are still included in sums.
summary_df4 = df3.groupby(
    ['YearMonth', 'DescriptionType', 'CancellationType'],
    dropna=False
).agg({
    'CustomerHashID': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

# 4. Rename the columns
summary_df4.columns = [
    'YearMonth', 
    'DescriptionType', 
    'CancellationType', 
    'Distinct_Customers', 
    'TotalPrice'
]


# 1. total revenue from df3
df3_total_revenue = df3['TotalPrice'].sum()
print(f"Base Revenue (df3): £{df3_total_revenue:,.2f}\n")

# 2. total revenue from summary_df3
summary_df4_total_revenue = summary_df4['TotalPrice'].sum()
print(f"Base Revenue (summary_df4): £{summary_df4_total_revenue:,.2f}\n")

output_csv_path4 = '../data/testing/summary_df4.csv'
print(f"⏳ Exporting test file to testing folder..")
summary_df4.to_csv(output_csv_path4, index=False)
print("✅ Export complete.")


# Display the first few rows
print(summary_df4.head())

# Check if there are any Customer Accounts missing an ID
rogue_nans = df1[df1['OrderCheckout'] == 'CUSTOMER ACCOUNT']['CustomerHashID'].isna().sum()
print(f"Number of Customer Accounts missing an ID: {rogue_nans}")
# Note I could just remove the .dropna(subset=['CustomerHashID']) 

Base Revenue (df3): £16,179,795.28

Base Revenue (summary_df4): £16,179,795.28

⏳ Exporting test file to testing folder..
✅ Export complete.
  YearMonth DescriptionType CancellationType  Distinct_Customers  TotalPrice
0   2009-12         DAMAGED              NaN                   1       20.85
1   2009-12        DISCOUNT     CANCELLATION                   5     -108.91
2   2009-12          MANUAL     CANCELLATION                   6    -2908.77
3   2009-12          MANUAL              NaN                   9     1403.19
4   2009-12         PRODUCT     CANCELLATION                 283   -17670.96
Number of Customer Accounts missing an ID: 0


---
##### We look at Cancellations, Damaged, Discount, Adjustment and Manual adjustments

We start with Manual in the stock code.

---
##### Check for same day manual cancellations where total price is opposite but quantity is different
These are where the sales assistant put manual cancellation quantity <= -1 to force total price to cancel but quantities differ. Wre need to stamp these as Manual Cancellation Re-Key and exclude, since whilst revenue will correctly net quantities won't. eg one has 60 items -1 cancel --> 59 is wrong

In [9]:
import pandas as pd
import numpy as np

# 1. Ensure we have a pure 'Date' column (stripping the hours/minutes)
# Using InvoiceDateMin or InvoiceDate, whichever has your timestamps
df3['OrderDate'] = pd.to_datetime(df3['InvoiceDate']).dt.date

# 2. Isolate the "Manual Fixes" (Negative TotalPrice, MANUAL description)
manual_fixes = df3[(df3['DescriptionType'] == 'MANUAL') & (df3['TotalPrice'] < 0)].copy()

# 3. Isolate "Product Orders" (Positive TotalPrice, regular products)
product_orders = df3[(df3['DescriptionType'] == 'PRODUCT') & (df3['TotalPrice'] > 0)].copy()

# 4. Create a matching key based on the absolute value of the money
# For the manual fix (-38970), we want to match it to a product of (+38970)
manual_fixes['MatchPrice'] = manual_fixes['TotalPrice'].abs()
product_orders['MatchPrice'] = product_orders['TotalPrice']

# 5. Merge them together to find perfect same-day matches
# We join on Customer, Date, and the Matching Price
same_day_offsets = pd.merge(
    product_orders, 
    manual_fixes, 
    on=['CustomerHashID', 'OrderDate', 'MatchPrice'], 
    suffixes=('_Orig', '_Manual')
)

# 6. Select columns to make it easy to read
columns_to_show =[
    'OrderDate', 
    'CustomerHashID',
    'Invoice_Orig', 'StockCode_Orig', 'Quantity_Orig', 'TotalPrice_Orig',
    'Invoice_Manual', 'Quantity_Manual', 'TotalPrice_Manual'
]

# Display the results
print(f"🔍 Found {len(same_day_offsets)} potential Same-Day Manual Offsets")
print(same_day_offsets[columns_to_show].sort_values('TotalPrice_Orig', ascending=False).head(20))

🔍 Found 16 potential Same-Day Manual Offsets
     OrderDate                                     CustomerHashID  \
6   2011-06-10  0269f3f8cf9733d029cb95cacc043bb01522d69d09972e...   
15  2011-11-22  16a1ca58c510a745d1eec451a4a41e1d17fe98058fd59d...   
4   2010-08-17  101f8a3aefa8b060a77022b45bec2dfe7a2775a996f24d...   
5   2010-08-17  101f8a3aefa8b060a77022b45bec2dfe7a2775a996f24d...   
0   2010-04-29  788be4bed3343b4750295b9c6d43c3b0d684b2b5932e34...   
1   2010-04-29  788be4bed3343b4750295b9c6d43c3b0d684b2b5932e34...   
12  2011-09-04  67df4c252659e16729dc399a7d39f3cf49923999206116...   
13  2011-09-04  67df4c252659e16729dc399a7d39f3cf49923999206116...   
14  2011-09-04  67df4c252659e16729dc399a7d39f3cf49923999206116...   
7   2011-07-01  43b2e2e8ddbf88a8ff47047daf73b834b9b7129df219c8...   
8   2011-07-01  43b2e2e8ddbf88a8ff47047daf73b834b9b7129df219c8...   
9   2011-07-01  43b2e2e8ddbf88a8ff47047daf73b834b9b7129df219c8...   
10  2011-07-01  43b2e2e8ddbf88a8ff47047daf73b834b9b7129df2

There are 171 Discounts, 167 of them are associated to a cancellation with negative quantity and total price. 4 of them are not associated to a cancelation and are positive quantity and price adjustment

---
##### We apply that rule as Decision 13 # remove Manual Cancellation Re-Keys

In [10]:
import pandas as pd
import numpy as np

# ==========================================
# STEP A: Pre-calculate Decision 13 Invoices
# ==========================================
# a. Define 'Order Date' column in df1
df1['OrderDate'] = pd.to_datetime(df1['InvoiceDateMin']).dt.date

# b. Isolate the "Manual Fixes" and "Product Orders"
manual_fixes = df1[(df1['DescriptionType'] == 'MANUAL') & (df1['TotalPrice'] < 0)].copy()
product_orders = df1[(df1['DescriptionType'] == 'PRODUCT') & (df1['TotalPrice'] > 0)].copy()

# c. Create matching key based on the absolute value of money
manual_fixes['MatchPrice'] = manual_fixes['TotalPrice'].abs()
product_orders['MatchPrice'] = product_orders['TotalPrice']

# d. Add counter to match 1-to-1
manual_fixes['MatchCount'] = manual_fixes.groupby(['CustomerHashID', 'OrderDate', 'MatchPrice']).cumcount()
product_orders['MatchCount'] = product_orders.groupby(['CustomerHashID', 'OrderDate', 'MatchPrice']).cumcount()

# e. Capture Row Index so we don't delete whole invoices
manual_fixes['RowIndex_Manual'] = manual_fixes.index
product_orders['RowIndex_Orig'] = product_orders.index

# f. Merge together on Date, Price, Customer and count
same_day_offsets = pd.merge(
    product_orders, 
    manual_fixes, 
    on=['CustomerHashID', 'OrderDate', 'MatchPrice', 'MatchCount']
)

# g. Create a list of row indices to delete
manual_cancellation_rekeys = list(same_day_offsets['RowIndex_Orig']) + list(same_day_offsets['RowIndex_Manual'])

# 1. Apply customer base filter rules
mask_df3 = (
    # Keep these:
    (df1['OrderCheckout'] == 'CUSTOMER ACCOUNT') & # Decision 1, 2, 3 remove test, warehouse and guest checkout
    (df1['InvoiceDateMin'] < '2011-12-01') & # Decisions 4, 5 # remove Dec 2011 
    
    # Exclude these:
    (~df1['CancellationType'].isin(['FULL INVOICE RE-KEY','LINE ITEM RE-KEY'])) & # Decisions 6, 6 # remove Re-Keys 
    (~df1['DescriptionType'].isin(['BANK CHARGES','CANCER RESEARCH COMMISSION'])) & # Decisions 8, 9 # remove Bank Charges and Charitable Donations 
    (~df1['DescriptionType'].isin(['CARRIAGE','DOTCOM POSTAGE','POSTAGE'])) & # Decisions 10, 11, 12 # remove Logistics Charges
    (~df1.index.isin(manual_cancellation_rekeys)) & # Decision 13 # remove Manual Cancellation Re-Keys
    (df1['UnitPrice'] != 0) # Decision 14 # remove Free Offers set at unit price 0
)

# 2. Apply mask to create new dataframe
df4 = df1.loc[mask_df3].dropna(subset=['CustomerHashID']).copy()

# 3. Group by YearMonth and two categorical columns
# dropna=False is included so orders with 'None' or empty values are still included in sums.
summary_df5 = df4.groupby(
    ['YearMonth', 'DescriptionType', 'CancellationType'],
    dropna=False
).agg({
    'CustomerHashID': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

# 4. Rename the columns
summary_df5.columns = [
    'YearMonth', 
    'DescriptionType', 
    'CancellationType', 
    'Distinct_Customers', 
    'TotalPrice'
]

df3_total_revenue = df3['TotalPrice'].sum()
print(f"Base Revenue (df3): £{df3_total_revenue:,.2f}\n")

# 1. total revenue from df4
df4_total_revenue = df4['TotalPrice'].sum()
print(f"Base Revenue (df4): £{df4_total_revenue:,.2f}\n")

# 2. total revenue from summary_df5
summary_df5_total_revenue = summary_df5['TotalPrice'].sum()
print(f"Base Revenue (summary_df5): £{summary_df5_total_revenue:,.2f}\n")

output_csv_path5 = '../data/testing/summary_df5.csv'
print(f"⏳ Exporting test file to testing folder..")
summary_df5.to_csv(output_csv_path5, index=False)
print("✅ Export complete.")


# Display the first few rows
print(summary_df5.head())

# Check if there are any Customer Accounts missing an ID
rogue_nans = df1[df1['OrderCheckout'] == 'CUSTOMER ACCOUNT']['CustomerHashID'].isna().sum()
print(f"Number of Customer Accounts missing an ID: {rogue_nans}")
# Note I could just remove the .dropna(subset=['CustomerHashID']) 

Base Revenue (df3): £16,179,795.28

Base Revenue (df4): £16,179,786.28

Base Revenue (summary_df5): £16,179,786.28

⏳ Exporting test file to testing folder..
✅ Export complete.
  YearMonth DescriptionType CancellationType  Distinct_Customers  TotalPrice
0   2009-12         DAMAGED              NaN                   1       20.85
1   2009-12        DISCOUNT     CANCELLATION                   5     -108.91
2   2009-12          MANUAL     CANCELLATION                   6    -2908.77
3   2009-12          MANUAL              NaN                   8     1403.19
4   2009-12         PRODUCT     CANCELLATION                 283   -17670.96
Number of Customer Accounts missing an ID: 0


In [12]:
# 1. Total Net Revenue includes EVERYTHING (Manuals, Discounts, Products)
revenue_calc = df3.groupby('CustomerHashID')['TotalPrice'].sum()

# 2. Total Product Items strictly EXCLUDES administrative rows
# We only sum quantities where it's a real product (Not Manual, Not Discount)
real_products_mask = ~df3['DescriptionType'].isin(['MANUAL', 'DISCOUNT'])
items_calc = df3[real_products_mask].groupby('CustomerHashID')['Quantity'].sum()

In [15]:
import pandas as pd

# 1. Investigate what Stock Codes are used for 'MANUAL' adjustments
manual_df = df4[df4['DescriptionType'] == 'MANUAL']

print("--- Top Stock Codes for MANUAL Adjustments ---")
# Assuming your original column is 'StockCode' (or change to 'StockCodeType' if needed)
if 'StockCode' in df4.columns:
    print(manual_df['StockCode'].value_counts().head(10))
else:
    print("StockCode column not found, check your exact column name.")

print("\n--- Summary of MANUAL Adjustments by Quantity ---")
# Let's see if manual adjustments have weird quantities that will mess up product counts
print(manual_df['Quantity'].describe())

print("\n--- Example MANUAL Rows ---")
# Look at a mix of positive and negative manual adjustments
print(manual_df.sort_values(by='TotalPrice').head(3)[['YearMonth', 'StockCode', 'Quantity', 'UnitPrice', 'TotalPrice']])
print(manual_df.sort_values(by='TotalPrice', ascending=False).head(3)[['YearMonth', 'StockCode', 'Quantity', 'UnitPrice', 'TotalPrice']])

# 2. Investigate Discounts
discount_df = df4[df4['DescriptionType'] == 'DISCOUNT']
print("\n--- Example DISCOUNT Rows ---")
print(discount_df[['YearMonth', 'StockCode', 'Quantity', 'UnitPrice', 'TotalPrice']].head())

--- Top Stock Codes for MANUAL Adjustments ---
StockCode
M    914
Name: count, dtype: int64

--- Summary of MANUAL Adjustments by Quantity ---
count     914.000000
mean        4.100656
std        93.261723
min     -1350.000000
25%        -1.000000
50%         1.000000
75%         3.000000
max      1600.000000
Name: Quantity, dtype: float64

--- Example MANUAL Rows ---
       YearMonth StockCode  Quantity  UnitPrice  TotalPrice
241824   2010-06         M        -1   25111.09   -25111.09
135014   2010-03         M        -1   10953.50   -10953.50
342135   2010-09         M        -1   10468.80   -10468.80
       YearMonth StockCode  Quantity  UnitPrice  TotalPrice
358639   2010-09         M         1   10468.80    10468.80
947838   2011-10         M         1    4161.06     4161.06
947835   2011-10         M         1    4161.06     4161.06

--- Example DISCOUNT Rows ---
      YearMonth StockCode  Quantity  UnitPrice  TotalPrice
735     2009-12         D        -1       9.00       -9.00


Okay, it is not many but there are examples of someone does a same day cancellation on an order and whilst the net revenue will cancel out, the quantities might not. In this instance it is better to isolate these as part of re-key adjustments:

<img src="images/manual_1.png" width="800">

Otherwise per the example below we want to net the manual revenue adjustment but not count quantity:

<img src="images/Manual_2.png" width="800">

---
# 7.0 Data Analysis and Findings
---
---